In [1]:
# Import libraries. 
import warnings
from math import log10, floor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as colours
import cmasher as cm
from astropy.io import ascii, fits
from astropy.table import Table, join
from scipy.odr import *

# Dismissal of Depreciation Warning for cleanliness
warnings.filterwarnings("ignore", category=DeprecationWarning, module="scipy.odr")
from scipy.odr import Model, RealData, ODR

C:\Users\troub\AppData\Local\Temp\ipykernel_18136\3782817206.py:11: DeprecationWarning: `scipy.odr` is deprecated as of version 1.17.0 and will be removed in SciPy 1.19.0. Please use `https://pypi.org/project/odrpack/` instead.
  from scipy.odr import *


In [ ]:
# Create table for final results.
# Kinematic outliers from the Kinematic-Analysis notebook
list= ["NGC362", "NGC1851", "NGC3201", "NGC6101", "NGC6218", "NGC6934", "NGC6981",
            "NGC1261", "NGC4147", "NGC5024", "NGC5053", "NGC5466", "NGC6715",
            "Arp2", "Pal12", "Terzan8", "NGC4147", "NGC4833", "NGC6584", "NGC6681"]
kinematic_outliers = set(list)

# AMR outliers from the AMR-Analysis notebook
vandenberg_outliers = {"NGC362", "NGC1261", "NGC6218", "NGC6362", "Pal12"}
krause_outliers = {"NGC362", "NGC1261", "NGC1851", "NGC6535", "Pal12"}

# Missing ages from the Krause data set
krause_missing = {"NGC4147", "Arp2", "Terzan8"}

def sort_key(name):
    digits = "".join(ch for ch in name if ch.isdigit())
    return (0, int(digits)) if name.startswith("NGC") else (1, name)

# Combine all outliers into a single list 
clusters = sorted(kinematic_outliers | vandenberg_outliers | krause_outliers, key=sort_key)
n = len(clusters)

def flag(cluster, outliers, missing=()):
    if cluster in missing:
        return "NA"
    return "✅" if cluster in outliers else "❌"

# Generate individual column lists first so we can reference them
vandenberg_flags = [flag(c, vandenberg_outliers) for c in clusters]
krause_flags = [flag(c, krause_outliers, krause_missing) for c in clusters]
kinematic_flags = [flag(c, kinematic_outliers) for c in clusters]

# Logic for Accretion Candidate: requires at least 2 checks ("✅")
def get_accretion_status(v, k, kin):
    # Count how many are strictly equal to "✅"
    checks_count = sum(1 for x in [v, k, kin] if x == "✅")
    return "✅" if checks_count >= 2 else "❌"

accretion_flags = [
    get_accretion_status(v, k, kin) 
    for v, k, kin in zip(vandenberg_flags, krause_flags, kinematic_flags)
]

results = pd.DataFrame({
    "AMR Outlier (VandenBerg)": vandenberg_flags,
    "AMR Outlier (Krause)": krause_flags,
    "Kinematic Outlier": kinematic_flags,
    "Strong Accretion Candidate": accretion_flags,
}, index=pd.Index(clusters, name="Cluster ID"))

# Decided to remove the "robust to catalogue choice" column as we are not contrasting results against other surveys as a part of this investigation. Thus, our results are the findings of the Harris, Krause, and VandenBerg data only.

results

,AMR Outlier (VandenBerg),AMR Outlier (Krause),Kinematic Outlier,Strong Accretion Candidate
Cluster ID,,,,
NGC362,✅,✅,✅,✅
NGC1261,✅,✅,✅,✅
NGC1851,❌,✅,✅,✅
NGC3201,❌,❌,✅,❌
NGC4147,❌,NA,✅,❌
NGC4833,❌,❌,✅,❌
NGC5024,❌,❌,✅,❌
NGC5053,❌,❌,✅,❌
NGC5466,❌,❌,✅,❌
